# Tree-Structured NB Regression on Spinal Cord Cross-Species Data

This notebook applies the tree-structured pseudobulk negative-binomial regression
to the multispecies spinal cord snRNA-seq dataset.

In [1]:
from __future__ import annotations
import torch
import sys
import warnings
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

warnings.filterwarnings('ignore')

# Add parent to path for imports
sys.path.insert(0, str(Path('.').resolve().parent.parent))
from tree_nb_regression import fit_tree_nb, build_taxonomy_tree_from_obs, build_species_tree_design

In [2]:
# ── Load Data ─────────────────────────────────────────────────────────────────
ADATA_PATH = Path(
    '/data/multispecies_integrated_realigned_qcfiltered_SpC_scvi_'
    'final_cluster_manual_annotations_mnqcfiltered.h5ad'
)
print(f'Loading {ADATA_PATH}...')
adata = ad.read_h5ad(ADATA_PATH)
print(f'Shape: {adata.shape}')
print(f'Species: {adata.obs["species"].value_counts().to_dict()}')

Loading /data/multispecies_integrated_realigned_qcfiltered_SpC_scvi_final_cluster_manual_annotations_mnqcfiltered.h5ad...


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/data/multispecies_integrated_realigned_qcfiltered_SpC_scvi_final_cluster_manual_annotations_mnqcfiltered.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
# ── Remove injury/treatment cells ─────────────────────────────────────────────
# Exclude spinal cord injury and drug treatment conditions, keeping only
# uninjured/control samples for cross-species DE analysis.
n_before = adata.n_obs

# GSE199669_injury:Macaque_mulatta — all are SCI, exclude entirely
injury_mask = adata.obs["study"] == "GSE199669_injury:Macaque_mulatta"

# GSE172167:Mouse — SCI timecourse, keep only batch=="Uninjured"
gse172_mask = (adata.obs["study"] == "GSE172167:Mouse") & (~adata.obs["batch"].str.contains("Uninjured"))

# GSE234774:Mouse — keep uninjured, age_old, sex_male; exclude all injury/drug conditions
gse234_donors = adata.obs["donor_name"].astype(str)
gse234_keep = (
    gse234_donors.str.startswith("uninjured")
    | gse234_donors.str.startswith("age_old")
    | gse234_donors.str.startswith("sex_male")
)
gse234_mask = (adata.obs["study"] == "GSE234774:Mouse") & (~gse234_keep)

# Combined exclusion mask
exclude_mask = injury_mask | gse172_mask | gse234_mask
print(f"Excluding {exclude_mask.sum():,} injury/treatment cells:")
print(f"  GSE199669_injury (macaque SCI): {injury_mask.sum():,}")
print(f"  GSE172167 (mouse SCI timepoints): {gse172_mask.sum():,}")
print(f"  GSE234774 (mouse injury/drug): {gse234_mask.sum():,}")

adata = adata[~exclude_mask].copy()
print(f"\nCells: {n_before:,} -> {adata.n_obs:,} ({n_before - adata.n_obs:,} removed)")

In [ ]:
# ── Use raw UMI counts ────────────────────────────────────────────────────────
adata.X = adata.layers['UMIs'].copy()
print(f'Using UMI counts layer, X type: {type(adata.X)}')
print(f'X sum sample: {adata.X[:5].sum()}')

In [ ]:
# ── Inspect taxonomy columns ──────────────────────────────────────────────────
taxonomy_cols = ['Neighborhood', 'Class_V2', 'Subclass_V2', 'Group_V2', 'final_cluster']
for col in taxonomy_cols:
    n = adata.obs[col].nunique()
    print(f'{col}: {n} unique values')

print(f'\nSpecies: {sorted(adata.obs["species"].unique())}')
print(f'Batches: {adata.obs["batch"].nunique()}')
print(f'Donors: {adata.obs["donor_name"].nunique()}')

In [ ]:
# ── Build and inspect taxonomy tree ───────────────────────────────────────────
tax_tree = build_taxonomy_tree_from_obs(adata.obs, taxonomy_cols)
print(f'Taxonomy nodes: {len(tax_tree.node_ids)}')
print(f'Taxonomy leaves: {len(tax_tree.leaf_ids)}')
print(f'Design matrix shape: {tax_tree.A_tax_leaf.shape}')
print(f'\nNode table head:')
tax_tree.node_table.head(10)

In [ ]:
# ── Build and inspect species tree ────────────────────────────────────────────
species_tree = '(Mouse,((Macaque_mulatta,Macaque_nemestrina),Human));'
observed_species = sorted(adata.obs['species'].unique().tolist())
print(f'Observed species: {observed_species}')

sp_design = build_species_tree_design(species_tree, observed_species)
print(f'Species design shape: {sp_design.A_species.shape}')
print(f'Species nodes: {sp_design.node_ids}')
print(f'\nPath indicators:')
pd.DataFrame(
    sp_design.A_species.toarray(),
    index=sp_design.species_order,
    columns=sp_design.node_ids
)

In [ ]:
# ── Fit model on subset of genes (demo) ───────────────────────────────────────
# Select HVGs from raw counts using variance-stabilization (seurat_v3),
# which operates on counts directly without per-cell log-normalization.
import scanpy as sc

sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=200, layer='UMIs')
hvg_genes = adata.var_names[adata.var['highly_variable']].tolist()
print(f'Selected {len(hvg_genes)} highly variable genes (seurat_v3 on raw counts)')

In [ ]:
# ── Subset to HVGs and fit (with tree-structured dispersion) ──────────────────
adata_sub = adata[:, hvg_genes].copy()
print(f'Fitting on {adata_sub.shape} (cells x genes)')

res = fit_tree_nb(
    adata_sub,
    taxonomy_cols=taxonomy_cols,
    species_col='species',
    species_tree=species_tree,
    batch_col='batch',
    donor_col='donor_name',
    gene_chunk_size=500,
    min_cells_per_pseudobulk=10,
    global_lambda=0.1,
    # NOTE: residual_lambda intentionally disabled when fitting dispersion —
    # gamma_ig and tree dispersion both explain unmodelled variance.
    residual_lambda=None,
    max_iter=1000,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    # ── Tree-structured dispersion (donor-level overdispersion per clade) ──
    fit_dispersion_tree=True,
    dispersion_lambda=0.3,
    min_replicates_per_node=3,
    # interactions intentionally omitted (under-replicated by default)
)
print('Fit complete!')
print(f'Diagnostics: {res.diagnostics}')

In [ ]:
# ── Examine results ───────────────────────────────────────────────────────────
print('Summary of nonzero coefficients per family:')
print(res.summary())
print(f'\nTotal pseudobulk groups: {res.diagnostics["n_groups"]}')
print(f'Total genes fit: {res.diagnostics["n_genes"]}')

In [ ]:
# ── Inspect taxonomy coefficients ─────────────────────────────────────────────
tax_coefs = res.get_coefficients_df('tax_global')
print(f'Taxonomy coefficients shape: {tax_coefs.shape}')
print(f'\nTop taxonomy effects (max abs across genes):')
max_effects = tax_coefs.abs().max(axis=1).sort_values(ascending=False)
print(max_effects.head(20))

In [ ]:
# ── Inspect species coefficients ──────────────────────────────────────────────
sp_coefs = res.get_coefficients_df('species_global')
print(f'Species coefficients shape: {sp_coefs.shape}')
print(f'\nSpecies effects (max abs across genes):')
print(sp_coefs.abs().max(axis=1).sort_values(ascending=False))

In [ ]:
# ── Check species-taxonomy interaction families ───────────────────────────────
for family in sorted(res.coefficients.keys()):
    if family.startswith('species_tax_'):
        coefs = res.coefficients[family]
        n_nonzero = (np.abs(coefs) > 0.01).sum()
        print(f'{family}: shape={coefs.shape}, nonzero={n_nonzero}')

---
## Diagnostic Plots: High-Magnitude Parameters & Expression Validation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from tree_nb_regression.pseudobulk import build_pseudobulk, aggregate_chunk

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200})

# Ensure hvg_genes matches adata_sub (in case subsetting was changed)
hvg_genes = list(adata_sub.var_names)

# ── Rebuild pseudobulk expression for plotting ────────────────────────────────
pb = build_pseudobulk(
    adata_sub.obs,
    taxonomy_col='final_cluster',
    species_col='species',
    batch_col='batch',
    donor_col='donor_name',
    min_cells_per_pseudobulk=10,
)
Y_pb = aggregate_chunk(adata_sub.X, pb.cell_to_group)
# Library-size normalize pseudobulk to log-CPM for plotting
lib_sizes = Y_pb.sum(axis=1, keepdims=True)
Y_logcpm = np.log1p(Y_pb / (lib_sizes + 1) * 1e4)

pb_expr = pd.DataFrame(Y_logcpm, columns=list(adata_sub.var_names))
pb_meta = pb.group_meta.copy()
print(f'Pseudobulk expression: {pb_expr.shape}')

In [ ]:
# ── Map coefficient indices back to taxonomy node labels ─────────────────────
tax_node_table = res.taxonomy_node_table
tax_node_ids = list(tax_node_table['node_id'].values)
tax_node_labels = list(tax_node_table['label'].values)
tax_node_levels = list(tax_node_table['level'].values)

sp_node_table = res.species_node_table
sp_node_ids = list(sp_node_table['node_id'].values)

### 1. Top Taxonomy Coefficients — Heatmap & Expression

In [ ]:
# ── Top taxonomy coefficients heatmap ─────────────────────────────────────────
tax_coefs = res.get_coefficients_df('tax_global')
# Pick top 20 nodes by max abs coefficient across genes
node_max = tax_coefs.abs().max(axis=1)
top_tax_nodes = node_max.nlargest(20).index.tolist()

# Annotate with level and label
top_tax_df = tax_coefs.loc[top_tax_nodes]
coef_idx_map = {f'tax_global_{i}': i for i in range(len(tax_node_ids))}
row_labels = []
for cid in top_tax_nodes:
    idx = coef_idx_map.get(cid, 0)
    if idx < len(tax_node_labels):
        row_labels.append(f'{tax_node_levels[idx]}:{tax_node_labels[idx]}')
    else:
        row_labels.append(cid)

fig, ax = plt.subplots(figsize=(min(16, len(hvg_genes) * 0.4), 7))
sns.heatmap(
    top_tax_df.values,
    yticklabels=row_labels,
    xticklabels=top_tax_df.columns,
    cmap='RdBu_r', center=0, ax=ax,
    cbar_kws={'label': 'Coefficient (log-scale effect)'},
)
ax.set_title('Top 20 Taxonomy Node Coefficients')
ax.set_xlabel('Gene')
ax.set_ylabel('Taxonomy Node')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# ── Expression boxplots for top taxonomy effects ──────────────────────────────
# For each of the top 5 taxonomy nodes, show expression of the gene with the
# largest coefficient, grouped by whether pseudobulk belongs to that node.

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for plot_idx, cid in enumerate(top_tax_nodes[:6]):
    idx = coef_idx_map.get(cid, 0)
    # Gene with max abs coef for this node
    gene_idx = int(np.argmax(np.abs(tax_coefs.loc[cid].values)))
    gene_name = hvg_genes[gene_idx]
    coef_val = tax_coefs.loc[cid].values[gene_idx]
    
    # Determine which pseudobulk groups belong to this node
    # A group belongs to a node if its taxonomy path includes that node
    node_id = tax_node_ids[idx] if idx < len(tax_node_ids) else ''
    level = tax_node_levels[idx] if idx < len(tax_node_levels) else ''
    label = tax_node_labels[idx] if idx < len(tax_node_labels) else ''
    
    # Match groups to this node via the taxonomy column
    if level in pb_meta.columns:
        in_node = pb_meta[level] == label
    else:
        in_node = pd.Series([False] * len(pb_meta))
    
    expr_in = pb_expr.loc[in_node.values, gene_name].values
    expr_out = pb_expr.loc[~in_node.values, gene_name].values
    
    ax = axes[plot_idx]
    parts = ax.violinplot([expr_in, expr_out], positions=[0, 1], showmedians=True)
    for pc in parts['bodies']:
        pc.set_alpha(0.6)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f'In {label}', f'Not in {label}'], fontsize=8)
    ax.set_ylabel(f'{gene_name} (log-CPM)')
    ax.set_title(f'{level}:{label}\ncoef={coef_val:.2f}', fontsize=9)

plt.suptitle('Expression of top taxonomy effects: In-node vs Out-of-node', fontsize=11)
plt.tight_layout()
plt.show()

### 2. Species Coefficients & Cross-Species Expression

In [ ]:
# ── Species coefficient bar chart ────────────────────────────────────────────
sp_coefs = res.get_coefficients_df('species_global')
# Show for top 6 genes with largest species effects
gene_sp_range = (sp_coefs.max(axis=0) - sp_coefs.min(axis=0)).sort_values(ascending=False)
top_sp_genes = gene_sp_range.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

for i, gene in enumerate(top_sp_genes):
    ax = axes[i]
    coefs_gene = sp_coefs[gene].values
    labels = [sp_node_ids[j] for j in range(len(coefs_gene))]
    colors = ['#2196F3' if c > 0 else '#F44336' for c in coefs_gene]
    ax.barh(range(len(coefs_gene)), coefs_gene, color=colors, alpha=0.7)
    ax.set_yticks(range(len(coefs_gene)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel('Coefficient')
    ax.set_title(gene, fontsize=9)

plt.suptitle('Species Tree Coefficients (top 6 genes by species range)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Expression by species for top species-effect genes ────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

species_order = sorted(adata_sub.obs['species'].unique())

for i, gene in enumerate(top_sp_genes):
    ax = axes[i]
    plot_data = []
    for sp in species_order:
        mask = pb_meta['species'] == sp
        vals = pb_expr.loc[mask.values, gene].values
        plot_data.append(vals)
    
    parts = ax.violinplot(plot_data, positions=range(len(species_order)), showmedians=True)
    for pc in parts['bodies']:
        pc.set_alpha(0.5)
    ax.set_xticks(range(len(species_order)))
    ax.set_xticklabels(species_order, fontsize=7, rotation=30)
    ax.set_ylabel('log-CPM')
    ax.set_title(gene, fontsize=9)

plt.suptitle('Pseudobulk expression by species (top species-effect genes)', fontsize=11)
plt.tight_layout()
plt.show()

### 3. Species × Taxonomy Interaction Effects

In [ ]:
# ── Top species-taxonomy deviation effects ───────────────────────────────────
# Find the strongest species-specific taxonomy deviations across all levels
interaction_effects = []
for family in sorted(res.coefficients.keys()):
    if not family.startswith('species_tax_'):
        continue
    coefs = res.coefficients[family]
    level_name = family.replace('species_tax_', '')
    for coef_idx in range(coefs.shape[0]):
        max_abs = np.max(np.abs(coefs[coef_idx]))
        best_gene_idx = int(np.argmax(np.abs(coefs[coef_idx])))
        if max_abs > 0.1:
            interaction_effects.append({
                'family': family,
                'level': level_name,
                'coef_idx': coef_idx,
                'max_abs': max_abs,
                'best_gene': hvg_genes[best_gene_idx],
                'best_gene_idx': best_gene_idx,
                'coef_value': coefs[coef_idx, best_gene_idx],
            })

int_df = pd.DataFrame(interaction_effects).sort_values('max_abs', ascending=False)
print(f'Species-taxonomy interactions with |coef| > 0.1: {len(int_df)}')
print(f'\nTop 15 species-taxonomy deviations:')
int_df.head(15)[['level', 'coef_idx', 'max_abs', 'best_gene', 'coef_value']]

In [ ]:
# ── Bar chart: number of strong interactions per taxonomy level ──────────────
level_counts = int_df.groupby('level').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
level_counts.plot.barh(ax=ax, color='steelblue', alpha=0.7)
ax.set_xlabel('Number of strong deviations (|coef| > 0.1)')
ax.set_ylabel('Taxonomy Level')
ax.set_title('Species-specific taxonomy deviations by level')
plt.tight_layout()
plt.show()

In [ ]:
# ── Expression plots for top species-taxonomy interactions ────────────────────
# For the top 6 interaction effects, show expression split by species AND taxonomy node

n_plot = min(6, len(int_df))
if n_plot > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.ravel()

    for plot_i in range(n_plot):
        row = int_df.iloc[plot_i]
        gene = row['best_gene']
        level = row['level']
        ax = axes[plot_i]
        
        # Determine which taxonomy nodes exist at this level
        if level in pb_meta.columns:
            level_vals = sorted(pb_meta[level].unique())
            # Show expression by species for each taxonomy value at this level
            # Pick the 2 most populated taxonomy values
            val_counts = pb_meta[level].value_counts()
            top_vals = val_counts.head(4).index.tolist()
            
            plot_rows = []
            for sp in species_order:
                for tv in top_vals:
                    mask = (pb_meta['species'] == sp) & (pb_meta[level] == tv)
                    vals = pb_expr.loc[mask.values, gene].values
                    if len(vals) > 0:
                        plot_rows.append({
                            'species': sp, level: tv,
                            'mean_expr': np.mean(vals),
                            'se': np.std(vals) / np.sqrt(len(vals)) if len(vals) > 1 else 0,
                        })
            
            if plot_rows:
                pdf = pd.DataFrame(plot_rows)
                # Grouped bar plot
                n_sp = len(species_order)
                n_tv = len(top_vals)
                x = np.arange(n_tv)
                width = 0.8 / n_sp
                for sp_i, sp in enumerate(species_order):
                    sp_data = pdf[pdf['species'] == sp]
                    means = [sp_data[sp_data[level] == tv]['mean_expr'].values[0]
                             if len(sp_data[sp_data[level] == tv]) > 0 else 0
                             for tv in top_vals]
                    ses = [sp_data[sp_data[level] == tv]['se'].values[0]
                           if len(sp_data[sp_data[level] == tv]) > 0 else 0
                           for tv in top_vals]
                    ax.bar(x + sp_i * width, means, width, yerr=ses,
                           label=sp, alpha=0.7, capsize=2)
                ax.set_xticks(x + width * (n_sp - 1) / 2)
                ax.set_xticklabels(top_vals, fontsize=6, rotation=30, ha='right')
                ax.legend(fontsize=6, loc='upper right')
        
        ax.set_ylabel(f'{gene} (log-CPM)')
        ax.set_title(f'{level} interaction\ncoef={row["coef_value"]:.2f}', fontsize=9)

    plt.suptitle('Top species × taxonomy interactions: mean expression by species & node', fontsize=11)
    plt.tight_layout()
    plt.show()

### 4. Predicted vs Observed Expression (Model Fit Quality)

In [ ]:
# ── Compute predicted values from fitted model ────────────────────────────────
from tree_nb_regression.model import _build_design_matrices
from tree_nb_regression.pseudobulk import build_pseudobulk
from tree_nb_regression.taxonomy_tree import build_taxonomy_tree_from_obs
from tree_nb_regression.species_tree import build_species_tree_design

# Rebuild designs
tax_tree = build_taxonomy_tree_from_obs(adata_sub.obs, taxonomy_cols)
sp_design = build_species_tree_design(species_tree, sorted(adata_sub.obs['species'].unique().tolist()))
pb_rebuild = build_pseudobulk(
    adata_sub.obs, taxonomy_col='final_cluster', species_col='species',
    batch_col='batch', donor_col='donor_name', min_cells_per_pseudobulk=10,
)
designs, _species_tax_meta_rebuild, _species_tax_node_groups_rebuild = _build_design_matrices(
    pb_rebuild, tax_tree, sp_design, taxonomy_cols, 'species', 'batch', 'donor_name',
)

# Compute library sizes
cell_totals = np.asarray(adata_sub.X.sum(axis=1)).ravel()
library_sizes = np.asarray((pb_rebuild.cell_to_group.T @ cell_totals)).ravel()
offset = np.log(library_sizes + 1e-8)

# Reconstruct linear predictor
eta = offset[:, None] + np.zeros((pb_rebuild.n_groups, len(hvg_genes)))
for family, X_design in designs.items():
    beta = res.coefficients[family]
    eta += X_design @ beta

# Add per-group residual intercepts if present
if res.gamma is not None:
    eta += res.gamma

mu_pred = np.exp(np.clip(eta, None, 20))
# Normalize predicted to log-CPM scale for comparison
mu_logcpm = np.log1p(mu_pred / (mu_pred.sum(axis=1, keepdims=True) + 1) * 1e4)
print(f'Predicted shape: {mu_pred.shape}')

In [ ]:
# ── Predicted vs Observed scatter plots ──────────────────────────────────────
# Pick 6 genes spanning different expression levels
gene_means = pb_expr.mean(axis=0).sort_values()
quantile_genes = [gene_means.index[int(q * (len(gene_means)-1))] for q in [0.1, 0.3, 0.5, 0.7, 0.85, 0.95]]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.ravel()

for i, gene in enumerate(quantile_genes):
    ax = axes[i]
    g_idx = hvg_genes.index(gene)
    obs_vals = Y_logcpm[:, g_idx]
    pred_vals = mu_logcpm[:, g_idx]
    
    # Subsample for readability if too many points
    n_pts = len(obs_vals)
    if n_pts > 2000:
        idx = np.random.default_rng(42).choice(n_pts, 2000, replace=False)
    else:
        idx = np.arange(n_pts)
    
    ax.scatter(obs_vals[idx], pred_vals[idx], s=3, alpha=0.2, rasterized=True)
    lims = [min(obs_vals[idx].min(), pred_vals[idx].min()),
            max(obs_vals[idx].max(), pred_vals[idx].max())]
    ax.plot(lims, lims, 'r--', lw=0.8, alpha=0.7)
    
    # Correlation
    from scipy.stats import pearsonr
    r, _ = pearsonr(obs_vals, pred_vals)
    ax.set_xlabel('Observed (log-CPM)')
    ax.set_ylabel('Predicted (log-CPM)')
    ax.set_title(f'{gene}\nr={r:.3f}', fontsize=9)

plt.suptitle('Predicted vs Observed Pseudobulk Expression', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Overall R² distribution across all genes ─────────────────────────────────
from scipy.stats import pearsonr

r_values = []
for g_idx in range(len(hvg_genes)):
    obs_g = Y_logcpm[:, g_idx]
    pred_g = mu_logcpm[:, g_idx]
    if np.std(obs_g) > 0 and np.std(pred_g) > 0:
        r, _ = pearsonr(obs_g, pred_g)
        r_values.append(r)
    else:
        r_values.append(0.0)

r_values = np.array(r_values)
r2_values = r_values ** 2

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(r_values, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(np.median(r_values), color='red', ls='--', label=f'median r={np.median(r_values):.3f}')
axes[0].set_xlabel('Pearson r (observed vs predicted)')
axes[0].set_ylabel('Number of genes')
axes[0].set_title('Prediction accuracy across genes')
axes[0].legend()

axes[1].hist(r2_values, bins=30, color='darkorange', alpha=0.7, edgecolor='white')
axes[1].axvline(np.median(r2_values), color='red', ls='--', label=f'median R²={np.median(r2_values):.3f}')
axes[1].set_xlabel('R² (variance explained)')
axes[1].set_ylabel('Number of genes')
axes[1].set_title('Variance explained across genes')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Pearson r: median={np.median(r_values):.3f}, mean={np.mean(r_values):.3f}')
print(f'R²: median={np.median(r2_values):.3f}, mean={np.mean(r2_values):.3f}')

### 5. Batch Effect Coefficients — Sanity Check

In [ ]:
# ── Batch coefficients distribution ──────────────────────────────────────────
if 'batch' in res.coefficients:
    batch_coefs = res.coefficients['batch']
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Distribution of all batch coefficients
    axes[0].hist(batch_coefs.ravel(), bins=50, color='gray', alpha=0.7, edgecolor='white')
    axes[0].axvline(0, color='red', ls='--', lw=0.8)
    axes[0].set_xlabel('Batch coefficient value')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'Batch coefficients distribution\n(shape: {batch_coefs.shape})')
    
    # Max abs batch effect per gene vs max taxonomy effect
    max_batch_per_gene = np.max(np.abs(batch_coefs), axis=0)
    tax_coefs_arr = res.coefficients['tax_global']
    max_tax_per_gene = np.max(np.abs(tax_coefs_arr), axis=0)
    
    axes[1].scatter(max_tax_per_gene, max_batch_per_gene, s=15, alpha=0.6)
    axes[1].plot([0, max(max_tax_per_gene.max(), max_batch_per_gene.max())],
                [0, max(max_tax_per_gene.max(), max_batch_per_gene.max())],
                'r--', lw=0.8)
    axes[1].set_xlabel('Max |taxonomy coef| per gene')
    axes[1].set_ylabel('Max |batch coef| per gene')
    axes[1].set_title('Batch vs Taxonomy effect magnitude')
    for i, gene in enumerate(hvg_genes):
        if max_batch_per_gene[i] > np.percentile(max_batch_per_gene, 90):
            axes[1].annotate(gene, (max_tax_per_gene[i], max_batch_per_gene[i]),
                           fontsize=6, alpha=0.7)
    
    plt.tight_layout()
    plt.show()

### 6. Coefficient Magnitude by Family — Overview

In [ ]:
# ── Coefficient magnitude summary across all families ─────────────────────────
family_stats = []
for family in sorted(res.coefficients.keys()):
    coefs = res.coefficients[family]
    abs_coefs = np.abs(coefs)
    family_stats.append({
        'family': family,
        'n_params': coefs.shape[0],
        'mean_abs': abs_coefs.mean(),
        'median_abs': np.median(abs_coefs),
        'max_abs': abs_coefs.max(),
        'pct_nonzero': (abs_coefs > 0.01).mean() * 100,
    })

stats_df = pd.DataFrame(family_stats)
print(stats_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(stats_df))
ax.bar(x, stats_df['max_abs'], alpha=0.4, label='Max |coef|', color='steelblue')
ax.bar(x, stats_df['mean_abs'], alpha=0.7, label='Mean |coef|', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(stats_df['family'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Coefficient magnitude')
ax.set_title('Coefficient magnitudes by parameter family')
ax.legend()
plt.tight_layout()
plt.show()

### 7. Coefficient Magnitudes on Tree Nodes — Taxonomy & Species Trees

In [ ]:
# ── Taxonomy tree: coefficient magnitude per node ─────────────────────────────
# For each taxonomy node, show the mean |coefficient| across genes,
# organized by tree level (depth).

tax_coefs_arr = res.coefficients['tax_global']  # (n_nodes, n_genes)
tax_nt = res.taxonomy_node_table.copy()
tax_nt['mean_abs_coef'] = np.abs(tax_coefs_arr).mean(axis=1)[:len(tax_nt)]
tax_nt['max_abs_coef'] = np.abs(tax_coefs_arr).max(axis=1)[:len(tax_nt)]

# Plot by level: strip plot of node magnitudes
levels_ordered = taxonomy_cols
tax_nt['level'] = pd.Categorical(tax_nt['level'], categories=levels_ordered, ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean |coef| per node, grouped by level
ax = axes[0]
for i, level in enumerate(levels_ordered):
    level_data = tax_nt[tax_nt['level'] == level]['mean_abs_coef'].values
    jitter = np.random.default_rng(42).uniform(-0.2, 0.2, len(level_data))
    ax.scatter(np.full(len(level_data), i) + jitter, level_data,
              s=15, alpha=0.5, color=f'C{i}')
    ax.plot([i - 0.3, i + 0.3], [np.median(level_data)] * 2,
            color='black', lw=2)
ax.set_xticks(range(len(levels_ordered)))
ax.set_xticklabels(levels_ordered, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Mean |coefficient| across genes')
ax.set_title('Taxonomy Tree: Mean coefficient magnitude per node')

# Max |coef| per node, grouped by level
ax = axes[1]
for i, level in enumerate(levels_ordered):
    level_data = tax_nt[tax_nt['level'] == level]['max_abs_coef'].values
    jitter = np.random.default_rng(42).uniform(-0.2, 0.2, len(level_data))
    ax.scatter(np.full(len(level_data), i) + jitter, level_data,
              s=15, alpha=0.5, color=f'C{i}')
    ax.plot([i - 0.3, i + 0.3], [np.median(level_data)] * 2,
            color='black', lw=2)
ax.set_xticks(range(len(levels_ordered)))
ax.set_xticklabels(levels_ordered, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Max |coefficient| across genes')
ax.set_title('Taxonomy Tree: Max coefficient magnitude per node')

plt.tight_layout()
plt.show()

# Print top nodes per level
print('\nTop 3 nodes per level by max |coef|:')
for level in levels_ordered:
    top = tax_nt[tax_nt['level'] == level].nlargest(3, 'max_abs_coef')[['label', 'max_abs_coef', 'mean_abs_coef']]
    print(f'  {level}:')
    for _, r in top.iterrows():
        print(f'    {r["label"]:30s}  max={r["max_abs_coef"]:.3f}  mean={r["mean_abs_coef"]:.3f}')

In [ ]:
# ── Taxonomy tree: heatmap of top nodes across genes ──────────────────────────
# Show the 30 nodes with highest max-coefficient, annotated by level

top30 = tax_nt.nlargest(30, 'max_abs_coef')
top30_idx = top30.index.tolist()
top30_coefs = tax_coefs_arr[top30_idx]

row_labels_tax = [f"{r['level']}:{r['label']}" for _, r in top30.iterrows()]

fig, ax = plt.subplots(figsize=(min(18, len(hvg_genes) * 0.35), 8))
im = ax.imshow(top30_coefs, aspect='auto', cmap='RdBu_r',
               vmin=-np.percentile(np.abs(top30_coefs), 95),
               vmax=np.percentile(np.abs(top30_coefs), 95))
ax.set_yticks(range(len(row_labels_tax)))
ax.set_yticklabels(row_labels_tax, fontsize=7)
ax.set_xticks(range(len(hvg_genes)))
ax.set_xticklabels(hvg_genes, rotation=90, fontsize=6)
ax.set_title('Taxonomy tree: Top 30 nodes by max coefficient magnitude')
plt.colorbar(im, ax=ax, label='Coefficient', shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Species tree: coefficient magnitude per node ──────────────────────────────

sp_coefs_arr = res.coefficients['species_global']  # (n_sp_nodes, n_genes)
sp_nt = res.species_node_table.copy()
sp_nt['mean_abs_coef'] = np.abs(sp_coefs_arr).mean(axis=1)[:len(sp_nt)]
sp_nt['max_abs_coef'] = np.abs(sp_coefs_arr).max(axis=1)[:len(sp_nt)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart: mean |coef| per species-tree node
ax = axes[0]
node_labels = sp_nt['node_id'].values
colors = ['#E91E63' if leaf else '#2196F3' for leaf in sp_nt['is_leaf']]
ax.barh(range(len(node_labels)), sp_nt['mean_abs_coef'].values, color=colors, alpha=0.7)
ax.set_yticks(range(len(node_labels)))
ax.set_yticklabels(node_labels, fontsize=9)
ax.set_xlabel('Mean |coefficient| across genes')
ax.set_title('Species Tree: Mean coef magnitude per node')
ax.legend(handles=[
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='#E91E63', markersize=10, label='Leaf (species)'),
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='#2196F3', markersize=10, label='Internal (clade)'),
], loc='lower right', fontsize=8)

# Bar chart: max |coef| per species-tree node
ax = axes[1]
ax.barh(range(len(node_labels)), sp_nt['max_abs_coef'].values, color=colors, alpha=0.7)
ax.set_yticks(range(len(node_labels)))
ax.set_yticklabels(node_labels, fontsize=9)
ax.set_xlabel('Max |coefficient| across genes')
ax.set_title('Species Tree: Max coef magnitude per node')

plt.tight_layout()
plt.show()

print('\nSpecies tree node coefficients:')
print(sp_nt[['node_id', 'is_leaf', 'mean_abs_coef', 'max_abs_coef']].to_string(index=False))

In [ ]:
# ── Species tree: full coefficient heatmap (nodes × genes) ───────────────────

fig, ax = plt.subplots(figsize=(min(18, len(hvg_genes) * 0.35), 3.5))
im = ax.imshow(sp_coefs_arr, aspect='auto', cmap='RdBu_r',
               vmin=-np.percentile(np.abs(sp_coefs_arr), 95),
               vmax=np.percentile(np.abs(sp_coefs_arr), 95))
ax.set_yticks(range(sp_coefs_arr.shape[0]))
ax.set_yticklabels(list(sp_nt['node_id'].values), fontsize=9)
ax.set_xticks(range(len(hvg_genes)))
ax.set_xticklabels(hvg_genes, rotation=90, fontsize=6)
ax.set_title('Species tree: Coefficients per node × gene')
plt.colorbar(im, ax=ax, label='Coefficient', shrink=0.9)
plt.tight_layout()
plt.show()

### 8. Cross-Species Divergence Score per Taxonomy Node

The taxonomy coefficients (section 7) show **cell-type specificity** shared across species.
The species coefficients show **global species offsets** across all cell types.

To identify **which taxonomy nodes are most divergent across species**, we need the
species × taxonomy interaction terms (`species_tax_*`). These capture where a species
deviates from the shared taxonomy pattern. Aggregating their magnitude per taxonomy
node gives a divergence score: nodes with high scores are cell types whose expression
programs differ most between species.

In [ ]:
# ── Compute divergence score per taxonomy node ─────────────────────────────────
# Use res.species_tax_meta for correct column→(species, node) mapping.
# Sum-to-zero parameterization: K columns per node (K = observed species at node).

sp_order = sorted(adata_sub.obs['species'].unique())
n_species = len(sp_order)

divergence_rows = []
for level in taxonomy_cols:
    family = f'species_tax_{level}'
    if family not in res.coefficients:
        continue
    coefs = res.coefficients[family]  # (n_cols, n_genes)
    meta_df = (res.species_tax_meta or {}).get(family)
    if meta_df is None:
        continue
    level_nodes = tax_nt[tax_nt['level'] == level]
    for _, node_row in level_nodes.iterrows():
        node_id = node_row['node_id']
        node_meta = meta_df[meta_df['node_id'] == node_id]
        col_indices = node_meta['col_index'].values
        if len(col_indices) == 0:
            continue
        node_coefs = coefs[col_indices]  # (K, n_genes)

        mean_divergence = np.abs(node_coefs).mean()
        max_divergence = np.abs(node_coefs).max()
        n_divergent_genes = int((np.abs(node_coefs).max(axis=0) > 0.05).sum())

        divergence_rows.append({
            'node_id': node_id,
            'label': node_row['label'],
            'level': level,
            'mean_divergence': mean_divergence,
            'max_divergence': max_divergence,
            'n_divergent_genes': n_divergent_genes,
        })

div_df = pd.DataFrame(divergence_rows)
div_df = div_df.sort_values('mean_divergence', ascending=False).reset_index(drop=True)
print(f'Computed divergence for {len(div_df)} taxonomy nodes')
print(f'\nTop 20 most divergent taxonomy nodes:')
print(div_df.head(20)[['level', 'label', 'mean_divergence', 'max_divergence', 'n_divergent_genes']].to_string(index=False))

In [ ]:
# ── Strip plot: divergence score by taxonomy level ────────────────────────────

div_df['level_cat'] = pd.Categorical(div_df['level'], categories=taxonomy_cols, ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for i, level in enumerate(taxonomy_cols):
    level_data = div_df[div_df['level'] == level]['mean_divergence'].values
    if len(level_data) == 0:
        continue
    jitter = np.random.default_rng(7).uniform(-0.25, 0.25, len(level_data))
    ax.scatter(np.full(len(level_data), i) + jitter, level_data,
              s=20, alpha=0.5, color=f'C{i}')
    ax.plot([i-0.3, i+0.3], [np.median(level_data)]*2, color='black', lw=2.5)
ax.set_xticks(range(len(taxonomy_cols)))
ax.set_xticklabels(taxonomy_cols, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Mean cross-species divergence')
ax.set_title('Divergence per taxonomy node\n(higher = more species-variable)')

ax = axes[1]
for i, level in enumerate(taxonomy_cols):
    level_data = div_df[div_df['level'] == level]['n_divergent_genes'].values
    if len(level_data) == 0:
        continue
    jitter = np.random.default_rng(7).uniform(-0.25, 0.25, len(level_data))
    ax.scatter(np.full(len(level_data), i) + jitter, level_data,
              s=20, alpha=0.5, color=f'C{i}')
    ax.plot([i-0.3, i+0.3], [np.median(level_data)]*2, color='black', lw=2.5)
ax.set_xticks(range(len(taxonomy_cols)))
ax.set_xticklabels(taxonomy_cols, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Number of divergent genes (|coef| > 0.05)')
ax.set_title('Breadth of divergence per taxonomy node')

plt.tight_layout()
plt.show()

In [ ]:
# ── Top divergent nodes: which species drive the divergence? ──────────────────
# Use res.species_tax_meta for correct column→(species, node) mapping.
# Each node shows all K observed species (sum-to-zero parameterization).

top_div = div_df.head(8)
n_plot = len(top_div)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.ravel()

for plot_i, (_, drow) in enumerate(top_div.iterrows()):
    level = drow['level']
    family = f'species_tax_{level}'
    coefs = res.coefficients[family]
    meta_df = (res.species_tax_meta or {}).get(family)
    ax = axes[plot_i]
    if meta_df is None:
        ax.set_visible(False)
        continue

    node_meta = meta_df[meta_df['node_id'] == drow['node_id']]
    sp_means = []
    sp_labels_plot = []
    for _, sp_row in node_meta.iterrows():
        sp_means.append(float(np.abs(coefs[sp_row['col_index']]).mean()))
        sp_labels_plot.append(sp_row['species'])

    ax.bar(range(len(sp_means)), sp_means,
           color=[f'C{j}' for j in range(len(sp_means))], alpha=0.7)
    ax.set_xticks(range(len(sp_labels_plot)))
    ax.set_xticklabels(sp_labels_plot, fontsize=7, rotation=30, ha='right')
    ax.set_ylabel('Mean |deviation from node mean|')
    ax.set_title(f'{level}:\n{drow["label"]}', fontsize=8)

for j in range(n_plot, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Top divergent taxonomy nodes: per-species deviations (sum-to-zero)\n'
    'Values are deviations from the node mean, not relative to a reference species.',
    fontsize=10,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Species tree divergence: which clades are most divergent? ─────────────────
# The species_global coefficient magnitude tells us which species-tree branches
# carry the most expression change. Large leaf coefficients = species-specific
# divergence. Large internal coefficients = clade-level divergence (shared shift
# across descendants relative to outgroup).

sp_coefs_arr = res.coefficients['species_global']
sp_nt_div = res.species_node_table.copy()
sp_nt_div['mean_abs_coef'] = np.abs(sp_coefs_arr).mean(axis=1)[:len(sp_nt_div)]
sp_nt_div['n_divergent_genes'] = (np.abs(sp_coefs_arr) > 0.05).sum(axis=1)[:len(sp_nt_div)]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#E91E63' if leaf else '#2196F3' for leaf in sp_nt_div['is_leaf']]
bars = ax.barh(range(len(sp_nt_div)), sp_nt_div['mean_abs_coef'].values,
               color=colors, alpha=0.7, edgecolor='white')
ax.set_yticks(range(len(sp_nt_div)))
labels = []
for _, r in sp_nt_div.iterrows():
    suffix = f" ({r['n_divergent_genes']}/{len(hvg_genes)} genes)"
    labels.append(r['node_id'] + suffix)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Mean |coefficient| (divergence from outgroup)')
ax.set_title('Species tree: Divergence per branch\n(leaf = species-specific; internal = clade-shared)')
ax.legend(handles=[
    plt.Line2D([0],[0], marker='s', color='w', markerfacecolor='#E91E63', markersize=10, label='Leaf (species-specific)'),
    plt.Line2D([0],[0], marker='s', color='w', markerfacecolor='#2196F3', markersize=10, label='Internal (clade-shared)'),
], loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('  - Large LEAF values = that species diverged in expression')
print('  - Large INTERNAL values = shared shift in a clade (e.g. primate vs mouse)')
print('  - Compare leaves: species with largest values are most divergent overall')

### 9. Group-Level Divergence Panel (Figure 3B–style)

Shows the mean cross-species divergence magnitude for each Group, with separate
markers for the Class-level, Subclass-level, and Group-level divergence.
Class/Subclass values are propagated to all descendant Groups.

In [ ]:
# ── Group divergence panel (Fig 3B style) ─────────────────────────────────────
import json as _json
from pathlib import Path as _Path
from matplotlib.lines import Line2D

# Load palettes and taxonomy lookup
ROOT = _Path('/scratch')
PAL_JSON = ROOT / 'intermediates' / 'figure1' / 'hierarchical_palettes.json'
TAX_TSV = ROOT / 'intermediates' / 'figure1' / 'taxonomy_group_summary.tsv'

with open(PAL_JSON) as f:
    PALETTES = _json.load(f)

# Use the category order from adata for Groups
GROUP_ORDER = list(adata.obs['Group_V2'].cat.categories)
GROUP_COLORS = PALETTES['Group_V2']['mapping']
_sg_colors = PALETTES.get('Supergroup_V2', {}).get('mapping', {})
_sc_colors = PALETTES.get('Subclass_V2', {}).get('mapping', {})

_tax = pd.read_csv(TAX_TSV, sep='\t')[['Group_V2','Supergroup_V2','Subclass_V2','Class_V2']].drop_duplicates('Group_V2')
_grp_to_sg = dict(zip(_tax['Group_V2'], _tax['Supergroup_V2']))
_grp_to_sc = dict(zip(_tax['Group_V2'], _tax['Subclass_V2']))
_grp_to_cl = dict(zip(_tax['Group_V2'], _tax['Class_V2']))

print(f'Groups in adata category order: {len(GROUP_ORDER)}')

In [ ]:
# ── Build divergence lookup per taxonomy node ─────────────────────────────────
# div_df was computed earlier; build a label→divergence dict per level
div_by_level = {}
for level in taxonomy_cols:
    sub = div_df[div_df['level'] == level]
    # Use dict to avoid Series.get() returning Series on dup keys
    div_by_level[level] = dict(zip(sub['label'], sub['mean_divergence']))

# For each Group in adata category order, look up divergence; filter unwanted groups
import re
_exclude_pat = re.compile(r'Monocyte|Myeloid|Macrophage|Lymph|Schwann|imm Oligo|ABC|VLMC|vMN')
plot_groups = [g for g in GROUP_ORDER
               if g in set(_tax['Group_V2']) and not _exclude_pat.search(str(g))]

group_div_class = np.array([
    float(div_by_level.get('Class_V2', {}).get(_grp_to_cl.get(g, ''), 0.0))
    for g in plot_groups
])
group_div_subclass = np.array([
    float(div_by_level.get('Subclass_V2', {}).get(_grp_to_sc.get(g, ''), 0.0))
    for g in plot_groups
])
group_div_group = np.array([
    float(div_by_level.get('Group_V2', {}).get(g, 0.0))
    for g in plot_groups
])

print(f'Plotting {len(plot_groups)} groups (after excluding Monocyte|Myeloid|Macrophage|Lymph|Schwann|imm Oligo|ABC|VLMC|vMN)')
print(f'Class divergence range: [{group_div_class.min():.3f}, {group_div_class.max():.3f}]')
print(f'Subclass divergence range: [{group_div_subclass.min():.3f}, {group_div_subclass.max():.3f}]')
print(f'Group divergence range: [{group_div_group.min():.3f}, {group_div_group.max():.3f}]')

In [ ]:
# ── Plot: Group divergence panel ──────────────────────────────────────────────
level_colors = {
    'Class': '#1976D2',
    'Subclass': '#F57C00',
    'Group': '#388E3C',
}

fig = plt.figure(figsize=(max(14, len(plot_groups) * 0.18), 6))
gs = fig.add_gridspec(4, 1, height_ratios=[4, 0.25, 0.25, 0.25], hspace=0.03)

ax = fig.add_subplot(gs[0])
ax_sg = fig.add_subplot(gs[1], sharex=ax)
ax_sc = fig.add_subplot(gs[2], sharex=ax)
ax_gr = fig.add_subplot(gs[3], sharex=ax)

x = np.arange(len(plot_groups))

# ── Class & Subclass as horizontal line segments (contiguous spans) ────────────
def _draw_level_lines(ax, plot_groups, group_to_node, div_lookup, color, linewidth=2.0, alpha=0.8):
    """Draw horizontal lines spanning contiguous runs of the same ancestor node."""
    if not plot_groups:
        return
    prev_node = group_to_node.get(plot_groups[0], '')
    run_start = 0
    for i in range(1, len(plot_groups) + 1):
        cur_node = group_to_node.get(plot_groups[i], '') if i < len(plot_groups) else None
        if cur_node != prev_node:
            # Draw line for this run
            val = float(div_lookup.get(prev_node, 0.0))
            ax.hlines(val, run_start - 0.4, i - 1 + 0.4,
                      colors=color, linewidth=linewidth, alpha=alpha, zorder=2)
            run_start = i
            prev_node = cur_node

_draw_level_lines(ax, plot_groups, _grp_to_cl,
                  div_by_level.get('Class_V2', {}), level_colors['Class'], linewidth=2.5)
_draw_level_lines(ax, plot_groups, _grp_to_sc,
                  div_by_level.get('Subclass_V2', {}), level_colors['Subclass'], linewidth=1.8)

# ── Group divergence as scatter points ─────────────────────────────────────────
ax.scatter(x, group_div_group, s=14, alpha=0.85, color=level_colors['Group'],
           zorder=3, edgecolors='none')

ax.set_xlim(-0.5, len(plot_groups) - 0.5)
ax.set_xticks(range(len(plot_groups)))
ax.set_xticklabels([])
ax.set_ylabel('Mean cross-species divergence')
ax.set_title('Cross-species divergence by Group (Class/Subclass propagated to descendants)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', linewidth=0.3, alpha=0.3)
ax.set_axisbelow(True)

legend_elements = [
    Line2D([0], [0], color=level_colors['Class'], linewidth=2.5, label='Class'),
    Line2D([0], [0], color=level_colors['Subclass'], linewidth=1.8, label='Subclass'),
    Line2D([0], [0], marker='o', color=level_colors['Group'], label='Group',
           markersize=6, linestyle='None'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

# ── Annotation bars ───────────────────────────────────────────────────────────
def _draw_annotation_bar(bar_ax, groups, color_lookup, label):
    for i, g in enumerate(groups):
        c = color_lookup.get(str(g), '#cccccc')
        bar_ax.barh(0, 1, left=i - 0.5, height=1, color=c, linewidth=0)
    bar_ax.set_xlim(-0.5, len(groups) - 0.5)
    bar_ax.set_ylim(0, 1)
    bar_ax.set_yticks([0.5])
    bar_ax.set_yticklabels([label], fontsize=6)
    bar_ax.tick_params(axis='y', length=0)
    for spine in bar_ax.spines.values():
        spine.set_visible(False)

_sg_lookup = {g: _sg_colors.get(str(_grp_to_sg.get(g, '')), '#cccccc') for g in plot_groups}
_sc_lookup = {g: _sc_colors.get(str(_grp_to_sc.get(g, '')), '#cccccc') for g in plot_groups}
_gr_lookup = {g: GROUP_COLORS.get(str(g), '#cccccc') for g in plot_groups}

_draw_annotation_bar(ax_sg, plot_groups, _sg_lookup, 'Supergroup')
_draw_annotation_bar(ax_sc, plot_groups, _sc_lookup, 'Subclass')
_draw_annotation_bar(ax_gr, plot_groups, _gr_lookup, 'Group')

ax_gr.set_xticks(range(len(plot_groups)))
ax_gr.set_xticklabels(plot_groups, rotation=90, fontsize=6)
ax_sg.tick_params(axis='x', labelbottom=False, length=0)
ax_sc.tick_params(axis='x', labelbottom=False, length=0)

plt.tight_layout()
plt.show()

### 9b. Per-Species/Clade Divergence by Group

Shows divergence broken out by species branch (Human, Macaque\_mulatta,
Macaque\_nemestrina vs Mouse reference) plus a Macaque-clade average.
Marker shapes distinguish taxonomy level (Class=square, Subclass=diamond, Group=circle).

In [ ]:
# ── Per-species divergence at each taxonomy level ─────────────────────────────
# Use res.species_tax_meta for correct column→(species, node) mapping.
# Sum-to-zero parameterization: deviation is relative to the node mean (not Mouse).

sp_order = sorted(adata.obs['species'].unique())
dev_species = sp_order[:-1]  # non-Mouse species for cross-species divergence plots
n_dev = len(dev_species)

# Build per-species, per-level divergence dicts: {level: {sp_idx: {node_label: div}}}
# NOTE: node_label is used as key for compatibility with downstream plot cells.
# Duplicate node_labels within a level will silently overwrite; use node_id for
# unambiguous lookups in new code.
div_per_species = {}  # level -> sp_idx -> {node_label: mean_abs_coef}
for level in taxonomy_cols:
    family = f'species_tax_{level}'
    if family not in res.coefficients:
        continue
    coefs = res.coefficients[family]
    level_nodes = tax_nt[tax_nt['level'] == level]
    meta_df = (res.species_tax_meta or {}).get(family)
    if meta_df is None:
        continue

    labels = set(level_nodes['label'].values)
    div_per_species[level] = {sp_idx: {} for sp_idx in range(n_dev)}
    for sp_idx, sp_name in enumerate(dev_species):
        sp_meta = meta_df[meta_df['species'] == sp_name]
        for _, sp_row in sp_meta.iterrows():
            node_label = sp_row['node_label']
            if node_label in labels:
                col_idx = sp_row['col_index']
                div_per_species[level][sp_idx][node_label] = float(
                    np.abs(coefs[col_idx]).mean()
                )

print(f'Per-species divergence computed for levels: {list(div_per_species.keys())}')
print(f'Deviation species: {dev_species} (reference: {sp_order[-1]})')

In [ ]:
# ── Build per-species divergence arrays for plot_groups ────────────────────────
# For each (level, species), get divergence at each group position
# Class/Subclass propagated to descendant Groups

level_to_grp_map = {
    'Class_V2': _grp_to_cl,
    'Subclass_V2': _grp_to_sc,
    'Group_V2': {g: g for g in plot_groups},  # identity
}

# Arrays: div_arrays[level][sp_idx] = np.array of length len(plot_groups)
div_arrays = {}
for level in ['Class_V2', 'Subclass_V2', 'Group_V2']:
    if level not in div_per_species:
        continue
    grp_map = level_to_grp_map[level]
    div_arrays[level] = {}
    for sp_idx in range(n_dev):
        sp_dict = div_per_species[level][sp_idx]
        vals = np.array([float(sp_dict.get(grp_map.get(g, ''), 0.0)) for g in plot_groups])
        div_arrays[level][sp_idx] = vals
    # Macaque clade = mean of Mac_mul (idx=1) and Mac_nem (idx=2)
    div_arrays[level]['macaque_clade'] = (
        div_arrays[level][1] + div_arrays[level][2]) / 2.0
    # Mouse branch = mean of all non-Mouse species (shared signal = Mouse is the outlier)
    div_arrays[level]['mouse_branch'] = (
        div_arrays[level][0] + div_arrays[level][1] + div_arrays[level][2]) / 3.0

print('Per-species arrays built for:', list(div_arrays.keys()))

In [ ]:
# ── Plot: per-species/clade divergence – separate panels per level ────────────
species_colors = {
    0: 'firebrick',              # Human
    1: 'mediumpurple',           # Macaque_mulatta
    2: 'dodgerblue',             # Macaque_nemestrina
    'macaque_clade': 'teal',     # Macaque clade (shared)
    'mouse_branch': 'goldenrod', # Mouse branch (all primates agree)
}
species_labels = {
    0: 'Human',
    1: 'Macaque mulatta',
    2: 'Macaque nemestrina',
    'macaque_clade': 'Macaque clade',
    'mouse_branch': 'Mouse branch',
}
sp_keys = [0, 1, 2, 'macaque_clade', 'mouse_branch']
sp_offsets = {0: -0.30, 1: -0.15, 2: 0.0, 'macaque_clade': 0.15, 'mouse_branch': 0.30}

levels_to_plot = [('Class_V2', 'Class'), ('Subclass_V2', 'Subclass'), ('Group_V2', 'Group')]

for level_key, level_name in levels_to_plot:
    if level_key not in div_arrays:
        continue

    fig = plt.figure(figsize=(max(14, len(plot_groups) * 0.18), 5))
    gs = fig.add_gridspec(4, 1, height_ratios=[4, 0.25, 0.25, 0.25], hspace=0.03)
    ax = fig.add_subplot(gs[0])
    ax_sg = fig.add_subplot(gs[1], sharex=ax)
    ax_sc = fig.add_subplot(gs[2], sharex=ax)
    ax_gr = fig.add_subplot(gs[3], sharex=ax)

    x = np.arange(len(plot_groups))
    for sp_key in sp_keys:
        vals = div_arrays[level_key][sp_key]
        ax.scatter(
            x + sp_offsets[sp_key], vals,
            s=18, alpha=0.8, marker='o',
            color=species_colors[sp_key],
            edgecolors='none', zorder=3,
        )

    ax.set_xlim(-0.5, len(plot_groups) - 0.5)
    ax.set_xticks(range(len(plot_groups)))
    ax.set_xticklabels([])
    ax.set_ylabel('Mean |coefficient|')
    ax.set_title(f'{level_name}-level divergence per species/clade')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', linewidth=0.3, alpha=0.3)
    ax.set_axisbelow(True)

    # Legend
    from matplotlib.lines import Line2D
    handles = [Line2D([0],[0], marker='o', color=species_colors[k],
                      label=species_labels[k], markersize=7, linestyle='None')
               for k in sp_keys]
    ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

    # Annotation bars
    def _draw_annotation_bar(bar_ax, groups, color_lookup, label):
        for i, g in enumerate(groups):
            c = color_lookup.get(str(g), '#cccccc')
            bar_ax.barh(0, 1, left=i - 0.5, height=1, color=c, linewidth=0)
        bar_ax.set_xlim(-0.5, len(groups) - 0.5)
        bar_ax.set_ylim(0, 1)
        bar_ax.set_yticks([0.5])
        bar_ax.set_yticklabels([label], fontsize=6)
        bar_ax.tick_params(axis='y', length=0)
        for spine in bar_ax.spines.values():
            spine.set_visible(False)

    _sg_lookup = {g: _sg_colors.get(str(_grp_to_sg.get(g, '')), '#cccccc') for g in plot_groups}
    _sc_lookup = {g: _sc_colors.get(str(_grp_to_sc.get(g, '')), '#cccccc') for g in plot_groups}
    _gr_lookup = {g: GROUP_COLORS.get(str(g), '#cccccc') for g in plot_groups}

    _draw_annotation_bar(ax_sg, plot_groups, _sg_lookup, 'Supergroup')
    _draw_annotation_bar(ax_sc, plot_groups, _sc_lookup, 'Subclass')
    _draw_annotation_bar(ax_gr, plot_groups, _gr_lookup, 'Group')

    ax_gr.set_xticks(range(len(plot_groups)))
    ax_gr.set_xticklabels(plot_groups, rotation=90, fontsize=6)
    ax_sg.tick_params(axis='x', labelbottom=False, length=0)
    ax_sc.tick_params(axis='x', labelbottom=False, length=0)

    plt.tight_layout()
    plt.show()

### 9c. Leaf-Level Total Divergence (global + cell-type-specific)

Adds the `species_global` tree-path contributions (internal nodes like the
macaque clade and primate stem) to the `species_tax` cell-type-specific
coefficients. This gives the **total** leaf-level divergence for each species
relative to Mouse at each taxonomy node.

In [ ]:
# ── Compute leaf-level total divergence (species_global path + species_tax) ──
# In sum-to-zero coding, beta_tax[species, node] is the species' deviation from
# the node mean — NOT relative to Mouse. To express divergence relative to Mouse,
# we use: total = global_path_diff(sp, Mouse) + (beta_tax[sp] - beta_tax[Mouse]).
# Nodes where Mouse is not observed are skipped (unidentifiable relative contrast).
from tree_nb_regression.species_tree import build_species_tree_design

sp_order = sorted(adata.obs['species'].unique())
sp_design_leaf = build_species_tree_design(
    '(Mouse,((Macaque_mulatta,Macaque_nemestrina),Human));', sp_order)
A = sp_design_leaf.A_species.toarray()  # (n_species, n_tree_nodes)
ref_idx = sp_order.index('Mouse')

beta_global = res.coefficients['species_global']  # (n_tree_nodes, n_genes)

# Path difference for each non-Mouse species relative to Mouse
path_diffs = {sp_idx: A[sp_idx] - A[ref_idx] for sp_idx in range(n_dev)}

leaf_div_per_species = {}  # level -> sp_idx -> {node_label: divergence}
for level in ['Class_V2', 'Subclass_V2', 'Group_V2']:
    family = f'species_tax_{level}'
    if family not in res.coefficients:
        continue
    coefs = res.coefficients[family]
    level_nodes = tax_nt[tax_nt['level'] == level]
    meta_df = (res.species_tax_meta or {}).get(family)
    if meta_df is None:
        continue

    labels = set(level_nodes['label'].values)
    leaf_div_per_species[level] = {sp_idx: {} for sp_idx in range(n_dev)}

    # Pre-build Mouse coefficient lookup per node_label: {node_label: coef_vec}
    mouse_meta = meta_df[meta_df['species'] == 'Mouse']
    mouse_coef = {
        row['node_label']: coefs[row['col_index']]
        for _, row in mouse_meta.iterrows()
        if row['node_label'] in labels
    }

    for sp_idx in range(n_dev):
        global_contrib = path_diffs[sp_idx] @ beta_global  # (n_genes,)
        sp_name = dev_species[sp_idx]
        sp_meta = meta_df[meta_df['species'] == sp_name]
        for _, sp_row in sp_meta.iterrows():
            node_label = sp_row['node_label']
            if node_label not in labels:
                continue
            # Skip nodes where Mouse is not observed (no relative contrast)
            if node_label not in mouse_coef:
                continue
            col_idx = sp_row['col_index']
            tax_contrib = coefs[col_idx]           # (n_genes,) — deviation from node mean
            mouse_tax = mouse_coef[node_label]     # (n_genes,) — Mouse deviation from node mean
            total = global_contrib + (tax_contrib - mouse_tax)
            leaf_div_per_species[level][sp_idx][node_label] = float(np.abs(total).mean())

print('Leaf-level total divergence computed for:', list(leaf_div_per_species.keys()))
print(f'Deviation species: {sp_order[:-1]} (reference: {sp_order[-1]})')

In [ ]:
# ── Build leaf-level arrays for plot_groups ───────────────────────────────────
leaf_arrays = {}
for level in ['Class_V2', 'Subclass_V2', 'Group_V2']:
    if level not in leaf_div_per_species:
        continue
    grp_map = level_to_grp_map[level]
    leaf_arrays[level] = {}
    for sp_idx in range(n_dev):
        sp_dict = leaf_div_per_species[level][sp_idx]
        leaf_arrays[level][sp_idx] = np.array(
            [float(sp_dict.get(grp_map.get(g, ''), 0.0)) for g in plot_groups])
    # Mouse = mean of all leaf divergences (Mouse's distance from the group)
    leaf_arrays[level]['mouse'] = (
        leaf_arrays[level][0] + leaf_arrays[level][1] + leaf_arrays[level][2]) / 3.0

print('Leaf arrays built. Sample ranges (Group_V2):')
for k, v in leaf_arrays.get('Group_V2', {}).items():
    lbl = sp_order[k] if isinstance(k, int) else k
    print(f'  {lbl}: [{v.min():.4f}, {v.max():.4f}]')

In [ ]:
# ── Plot: leaf-level total divergence – separate panels per level ────────────
leaf_colors = {
    0: 'firebrick',       # Human
    1: 'mediumpurple',    # Macaque_mulatta
    2: 'dodgerblue',     # Macaque_nemestrina
    'mouse': 'goldenrod', # Mouse (mean of others)
}
leaf_labels = {
    0: 'Human',
    1: 'Macaque mulatta',
    2: 'Macaque nemestrina',
    'mouse': 'Mouse',
}
leaf_keys = [0, 1, 2, 'mouse']
leaf_offsets = {0: -0.25, 1: -0.08, 2: 0.08, 'mouse': 0.25}

levels_to_plot = [('Class_V2', 'Class'), ('Subclass_V2', 'Subclass'), ('Group_V2', 'Group')]

for level_key, level_name in levels_to_plot:
    if level_key not in leaf_arrays:
        continue

    fig = plt.figure(figsize=(max(14, len(plot_groups) * 0.18), 5))
    gs = fig.add_gridspec(4, 1, height_ratios=[4, 0.25, 0.25, 0.25], hspace=0.03)
    ax = fig.add_subplot(gs[0])
    ax_sg = fig.add_subplot(gs[1], sharex=ax)
    ax_sc = fig.add_subplot(gs[2], sharex=ax)
    ax_gr = fig.add_subplot(gs[3], sharex=ax)

    x = np.arange(len(plot_groups))
    for sp_key in leaf_keys:
        vals = leaf_arrays[level_key][sp_key]
        ax.scatter(
            x + leaf_offsets[sp_key], vals,
            s=18, alpha=0.8, marker='o',
            color=leaf_colors[sp_key],
            edgecolors='none', zorder=3,
        )

    ax.set_xlim(-0.5, len(plot_groups) - 0.5)
    ax.set_xticks(range(len(plot_groups)))
    ax.set_xticklabels([])
    ax.set_ylabel('Mean |total leaf coefficient|')
    ax.set_title(f'{level_name}-level: total leaf divergence (global path + cell-type-specific)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', linewidth=0.3, alpha=0.3)
    ax.set_axisbelow(True)

    from matplotlib.lines import Line2D
    handles = [Line2D([0],[0], marker='o', color=leaf_colors[k],
                      label=leaf_labels[k], markersize=7, linestyle='None')
               for k in leaf_keys]
    ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

    # Annotation bars
    def _draw_annotation_bar(bar_ax, groups, color_lookup, label):
        for i, g in enumerate(groups):
            c = color_lookup.get(str(g), '#cccccc')
            bar_ax.barh(0, 1, left=i - 0.5, height=1, color=c, linewidth=0)
        bar_ax.set_xlim(-0.5, len(groups) - 0.5)
        bar_ax.set_ylim(0, 1)
        bar_ax.set_yticks([0.5])
        bar_ax.set_yticklabels([label], fontsize=6)
        bar_ax.tick_params(axis='y', length=0)
        for spine in bar_ax.spines.values():
            spine.set_visible(False)

    _sg_lookup = {g: _sg_colors.get(str(_grp_to_sg.get(g, '')), '#cccccc') for g in plot_groups}
    _sc_lookup = {g: _sc_colors.get(str(_grp_to_sc.get(g, '')), '#cccccc') for g in plot_groups}
    _gr_lookup = {g: GROUP_COLORS.get(str(g), '#cccccc') for g in plot_groups}

    _draw_annotation_bar(ax_sg, plot_groups, _sg_lookup, 'Supergroup')
    _draw_annotation_bar(ax_sc, plot_groups, _sc_lookup, 'Subclass')
    _draw_annotation_bar(ax_gr, plot_groups, _gr_lookup, 'Group')

    ax_gr.set_xticks(range(len(plot_groups)))
    ax_gr.set_xticklabels(plot_groups, rotation=90, fontsize=6)
    ax_sg.tick_params(axis='x', labelbottom=False, length=0)
    ax_sc.tick_params(axis='x', labelbottom=False, length=0)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Save outputs for downstream divergence analysis ──────────────────────────
import joblib
import xarray as xr
from pathlib import Path

OUTDIR = Path('/results/tree_nb_regression')
OUTDIR.mkdir(parents=True, exist_ok=True)

# 0. ── Save the full TreeNBResult object for downstream reuse ─────────────────
# This allows reloading res without re-fitting: joblib.load(OUTDIR / 'tree_nb_result.pkl')
RES_PATH = OUTDIR / 'tree_nb_result.pkl'
joblib.dump(res, RES_PATH, compress=3)
print(f'Saved TreeNBResult -> {RES_PATH}')

# 1. ── Save gene-level species_tax coefficients as xarray Dataset ─────────────
#    Dimensions: (species, node_{level}, gene).
#    Parameterization: sum-to-zero (deviation from node mean; NaN = not observed).
#    Use node_id as the node coordinate (unambiguous, no duplicate-label risk).
all_species_list = sorted(adata_sub.obs['species'].unique().tolist())
data_vars = {}
mask_vars = {}
for level in taxonomy_cols:
    family = f'species_tax_{level}'
    if family not in res.coefficients:
        continue
    coefs = res.coefficients[family]
    mask = res.selected_nonzero[family]
    meta_df = (res.species_tax_meta or {}).get(family)
    if meta_df is None:
        continue

    level_nodes = tax_nt[tax_nt['level'] == level].copy()
    node_ids = list(level_nodes['node_id'].values)
    n_level_nodes = len(node_ids)
    n_sp = len(all_species_list)
    n_genes_fit = len(res.gene_names)

    coef_arr = np.full((n_sp, n_level_nodes, n_genes_fit), np.nan)
    mask_arr = np.zeros((n_sp, n_level_nodes, n_genes_fit), dtype=bool)

    sp_to_idx = {sp: i for i, sp in enumerate(all_species_list)}
    node_id_to_idx = {nid: i for i, nid in enumerate(node_ids)}

    for _, mrow in meta_df.iterrows():
        sp_i = sp_to_idx.get(mrow['species'])
        node_i = node_id_to_idx.get(mrow['node_id'])
        if sp_i is None or node_i is None:
            continue
        col_idx = mrow['col_index']
        coef_arr[sp_i, node_i, :] = coefs[col_idx]
        mask_arr[sp_i, node_i, :] = mask[col_idx]

    dims = ['species', f'node_{level}', 'gene']
    coords = {
        'species': all_species_list,
        f'node_{level}': node_ids,
        'gene': res.gene_names,
    }
    data_vars[level] = xr.DataArray(
        coef_arr, dims=dims, coords=coords,
        attrs={'level': level, 'parameterization': 'sum_to_zero_deviation_from_node_mean'},
    )
    mask_vars[level] = xr.DataArray(
        mask_arr, dims=dims, coords=coords,
        attrs={'level': level},
    )

gene_div_ds = xr.Dataset(data_vars)
gene_div_ds.to_netcdf(OUTDIR / 'gene_level_divergence.nc')

# 1b. L1-selection masks
gene_div_mask_ds = xr.Dataset(mask_vars)
gene_div_mask_ds.to_netcdf(OUTDIR / 'gene_level_divergence_selected.nc')

# 2. Save div_df (per-node divergence summary)
div_df.to_csv(OUTDIR / 'node_divergence_summary.csv', index=False)

# 3-4. Save global coefficients AND their selection masks
sp_global_df = res.get_coefficients_df('species_global')
sp_global_df.to_csv(OUTDIR / 'species_global_coefficients.csv')
pd.DataFrame(
    res.selected_nonzero['species_global'],
    index=sp_global_df.index,
    columns=sp_global_df.columns,
).to_csv(OUTDIR / 'species_global_selected.csv')

tax_global_df = res.get_coefficients_df('tax_global')
tax_global_df.to_csv(OUTDIR / 'taxonomy_global_coefficients.csv')
pd.DataFrame(
    res.selected_nonzero['tax_global'],
    index=tax_global_df.index,
    columns=tax_global_df.columns,
).to_csv(OUTDIR / 'taxonomy_global_selected.csv')

# 5. Save taxonomy/species node tables
tax_nt.to_csv(OUTDIR / 'taxonomy_node_table.csv', index=False)
res.species_node_table.to_csv(OUTDIR / 'species_node_table.csv', index=False)

# 6. Save the taxonomy mapping
_tax.to_csv(OUTDIR / 'taxonomy_mapping.csv', index=False)

# 7. Save plot_groups list
pd.DataFrame({'Group_V2': plot_groups}).to_csv(OUTDIR / 'plot_groups_order.csv', index=False)

# 8. ── Dispersion outputs (tree-structured pseudobulk dispersion fit) ────────
if res.dispersion_coefficients is not None:
    DISP_DIR = OUTDIR / 'dispersion'
    DISP_DIR.mkdir(exist_ok=True)
    for fam in res.dispersion_coefficients:
        df_disp = res.get_dispersion_df(fam)
        df_disp.to_csv(DISP_DIR / f'{fam}_log_overdispersion.csv')
        sel = pd.DataFrame(
            res.dispersion_selected_nonzero[fam],
            index=df_disp.index,
            columns=df_disp.columns,
        )
        sel.to_csv(DISP_DIR / f'{fam}_selected.csv')
        for thr in (0.1, 0.2, 0.5):
            calls = res.call_dispersion(fam, threshold=thr)
            calls.to_csv(DISP_DIR / f'{fam}_calls_thr{thr}.csv')
    res.dispersion_coef_metadata.to_csv(DISP_DIR / 'coef_metadata.csv', index=False)
    print(f'Dispersion outputs -> {DISP_DIR}')
    print(f"  Selected per family: {res.diagnostics.get('dispersion_nonzero_per_family')}")
    print(f"  Active cols per family: {res.diagnostics.get('dispersion_active_cols_per_family')}")
else:
    print('No dispersion fit was performed (fit_dispersion_tree=False).')

print(f'\nSaved tree_nb outputs to {OUTDIR}:')
for f in sorted(OUTDIR.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(OUTDIR)} ({f.stat().st_size/1024:.1f} KB)')


# 9. ── Dispersion-aware Wald significance for L1-selected mean coefs ────────
from tree_nb_regression.inference import compute_wald_significance
print('\nComputing Wald significance for selected coefficients...')
wald_df = compute_wald_significance(
    res,
    add_q=True,
)
WALD_PATH = OUTDIR / 'wald_significance.parquet'
try:
    wald_df.to_parquet(WALD_PATH, index=False)
    print(f'  Saved {len(wald_df)} rows -> {WALD_PATH}')
except Exception as _e:
    WALD_PATH = OUTDIR / 'wald_significance.csv'
    wald_df.to_csv(WALD_PATH, index=False)
    print(f'  parquet failed ({_e}); saved CSV -> {WALD_PATH}')
print('  q<0.05 selected fraction per family:')
for fam, sub in wald_df.groupby('family'):
    print(f'    {fam}: {(sub["q"] < 0.05).mean():.3%} of {len(sub)} selected coefs')